# 02_extract_dhw_at_survey

### Downloading *Daily Global 5km Satellite Coral Bleaching Heat Stress Degree Heating Week*
### Associate the corresponding daily DHW value to each bleaching report with an exact date

#### Import libraries

In [1]:
import pandas as pd
import numpy as np
import xarray as xr
import requests
import os
from datetime import datetime, timedelta
from tqdm import tqdm
import gc
import time
from pathlib import Path

#### Import raw database

In [ ]:
# Go up one level from notebooks/ to 01_data_assembly/
root = Path("..")
file_path_import = root / "data" / "raw" / "Bleaching Database V2 - Urcelay and Donner.xlsx"
df = pd.read_excel(file_path_import)

#### Create clean dataset with exact dates
This step filters the dataset to retain only entries with complete and valid date information.

In [3]:
df_exact = df.copy()

for col in ['YEAR', 'MONTH', 'DATE']:
    df_exact[col] = pd.to_numeric(df_exact[col], errors='coerce')

df_exact = df_exact[
    df_exact['YEAR'].notnull() &
    df_exact['MONTH'].notnull() &
    df_exact['DATE'].notnull()
].copy()

df_exact['YEAR'] = df_exact['YEAR'].astype(int)
df_exact['MONTH'] = df_exact['MONTH'].astype(int)

df_exact = df_exact.rename(columns={'DATE': 'DAY'})
df_exact['DAY'] = df_exact['DAY'].astype(int)

print(f"Number of entries with exact dates: {len(df_exact)}")

Number of entries with exact dates: 21008


#### Create the datetime column

In [4]:
df_exact['DATETIME'] = pd.to_datetime(df_exact[['YEAR', 'MONTH', 'DAY']])
print(df_exact[['YEAR', 'MONTH', 'DAY', 'DATETIME']].head())

     YEAR  MONTH  DAY   DATETIME
358  1993     10   11 1993-10-11
384  1994      5   14 1994-05-14
390  1994      9   25 1994-09-25
477  1995      5   14 1995-05-14
495  1995     10    8 1995-10-08


#### Create a Column for DHW Values

In [5]:
df_exact['DHW'] = None

#### Making sure there is no missing GPS coordinate values

In [7]:
missing_coords = df_exact[
    df_exact["LATITUDE"].isna() | df_exact["LONGITUDE"].isna()
]

print(f"Rows missing coordinates: {len(missing_coords)}")
display(missing_coords)

Rows missing coordinates: 3


,FID,OCEAN_REGION,COUNTRY,LOCATION,SITE_NAME,LATITUDE,LONGITUDE,DAY,MONTH,YEAR,...,CRW DAILY DHW,CRW ANNUAL MAX DHW,CRW DHW OFFSET,QA CODE,CRW ANN MAX DHW 2014,CRW ANN MAX DHW 2015,CRW ANN MAX DHW 2016,CRW ANN MAX DHW 2017,DATETIME,DHW
23640,23641,NaN,French Polynesia,Moorea,NaN,NaN,NaN,2,3,2015,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015-03-02,None
23641,23642,NaN,French Polynesia,Moorea,NaN,NaN,NaN,2,3,2015,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015-03-02,None
23887,23888,NaN,Malaysia,"Banggi, Sabah",Batuan,NaN,NaN,21,4,2015,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015-04-21,None


#### Entering values manually when feasible

In [8]:
manual_coords = {
    23641: {"LATITUDE": -17.54, "LONGITUDE": -149.83},
    23642: {"LATITUDE": -17.54, "LONGITUDE": -149.83},
    23888: {"LATITUDE": 7.26, "LONGITUDE": 117.15},
}

for fid, coords in manual_coords.items():
    mask = df_exact["FID"] == fid

    if mask.sum() != 1:
        print(f"Warning: FID {fid} matched {mask.sum()} rows")

    df_exact.loc[mask, "LATITUDE"] = coords["LATITUDE"]
    df_exact.loc[mask, "LONGITUDE"] = coords["LONGITUDE"]

In [9]:
df_exact[df_exact["FID"].isin([23641, 23642, 23888])][
    ["FID", "LATITUDE", "LONGITUDE"]
]

,FID,LATITUDE,LONGITUDE
23640,23641,-17.54,-149.83
23641,23642,-17.54,-149.83
23887,23888,7.26,117.15


#### - Extracts DHW from NOAA CRW NetCDF files per row in df_exact.
#### - Files are cached in temp_nc/.
#### - Script reprocesses all rows each run for reproducibility.
#### - Delete cache to force updated NOAA values

##### open_verified_dataset function

In [2]:
def is_valid_file(path, min_size_kb=100):
    return os.path.exists(path) and os.path.getsize(path) > min_size_kb * 1024


def open_verified_dataset(local_file, url, max_retries=2):
    for attempt in range(max_retries):
        try:
            if not is_valid_file(local_file):
                raise OSError("File is too small or empty.")
            ds = xr.open_dataset(local_file, engine ="netcdf4")
            return ds
        except Exception as e:
            print(f"Corrupted or unreadable file: {local_file} — {e}")
            try:
                os.remove(local_file)
                print(f"Deleted corrupted file: {local_file}")
            except Exception as delete_error:
                print(f"Could not delete file: {delete_error}")
                return None
            print(f"Attempting to redownload {local_file}")
            try:
                r = requests.get(url, timeout=10)
                if r.status_code == 200:
                    with open(local_file, "wb") as f:
                        f.write(r.content)
                    time.sleep(0.1)
                else:
                    print(f"Download failed: HTTP {r.status_code}")
                    return None
            except Exception as dl_error:
                print(f"Download error: {dl_error}")
                return None
    print(f"Failed to open NetCDF after {max_retries} attempts: {local_file}")
    return None

##### get_valid_dhw function

In [3]:
def get_valid_dhw(ds, lat, lon, search_radius=10):
    try:
        dhw = ds['degree_heating_week'].isel(time=0)
        lats = ds['lat'].values
        lons = ds['lon'].values

        lat_idx = np.abs(lats - lat).argmin()
        lon_idx = np.abs(lons - lon).argmin()

        lat_start = max(lat_idx - search_radius, 0)
        lat_end = lat_idx + search_radius + 1
        lon_start = max(lon_idx - search_radius, 0)
        lon_end = lon_idx + search_radius + 1

        sub_dhw = dhw[lat_start:lat_end, lon_start:lon_end]

        closest_val = None
        min_dist = np.inf

        for i in range(sub_dhw.shape[0]):
            for j in range(sub_dhw.shape[1]):
                val = sub_dhw.values[i, j]
                if not np.isnan(val):
                    grid_lat = lats[lat_start + i]
                    grid_lon = lons[lon_start + j]
                    dist = np.sqrt((lat - grid_lat)**2 + (lon - grid_lon)**2)
                    if dist < min_dist:
                        min_dist = dist
                        closest_val = val

        return float(closest_val) if closest_val is not None else None

    except Exception:
        return None

##### process_row function

In [4]:
def process_row(idx_row):
    idx, row = idx_row
    try:
        date_str = row['DATETIME'].strftime('%Y%m%d')
        year_str = row['DATETIME'].strftime('%Y')
        lat = row['LATITUDE']
        lon = row['LONGITUDE']
        
        local_file = f"temp_nc/dhw_{date_str}.nc"
        url = f"https://www.star.nesdis.noaa.gov/pub/socd/mecb/crw/data/5km/v3.1_op/nc/v1.0/daily/dhw/{year_str}/ct5km_dhw_v3.1_{date_str}.nc"
        
        if not os.path.exists(local_file):
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                with open(local_file, "wb") as f:
                    f.write(r.content)
                time.sleep(0.1)
            else:
                print(f"Could not download file for index {idx}, status code {r.status_code}")
                return (idx, None)

        ds = open_verified_dataset(local_file, url)
        if ds is None:
            return(idx,None)  
            
        try:            
            val = ds['degree_heating_week'].sel(lat=lat,lon=lon,method='nearest').values.item()
            if np.isnan(val):
                val = get_valid_dhw(ds, lat, lon, search_radius=10)
        finally:
            ds.close()
            
        gc.collect()     
        return (idx, val)

    except Exception as e:
        print(f"Error at index {idx}: {e}")
        return (idx, None)

In [8]:
os.makedirs("temp_nc", exist_ok=True)
                        
results = []
for item in tqdm(df_exact.iterrows(), total=len(df_exact), desc="Processing DHW (sequential)"):
    result = process_row(item)
    results.append(result)

for idx, dhw_val in results:
    df_exact.at[idx, 'DHW'] = dhw_val

Processing DHW (sequential): 100%|█████████████████████████████████████████████| 21008/21008 [1:32:36<00:00,  3.78it/s]


#### Exporting expanded database with DHW values as df_exact_dhw.xlsx

In [9]:
file_path_export = root / "data" / "intermediate" / "df_exact_dhw.xlsx"

file_path_export.parent.mkdir(parents=True, exist_ok=True)

df_exact.to_excel(file_path_export, index=False)